In [0]:
display(spark.sql("""
    SELECT DISTINCT product_category 
    FROM fraud_detection.bronze.transactions_raw
    ORDER BY product_category
"""))

In [0]:
display(dbutils.fs.ls("/Volumes/fraud_detection/bronze/raw_files/"))

In [0]:
from pyspark.sql.functions import trim, regexp_replace, col

product_df = spark.read.option("header", "true").option("inferSchema", "true").option("quote", "").csv(
    "/Volumes/fraud_detection/bronze/raw_files/product_catalog.csv"
)

# Clean column names by removing quotes
for old_col in product_df.columns:
    new_col = old_col.replace('"', '')
    product_df = product_df.withColumnRenamed(old_col, new_col)

# Clean quote marks from string values
for col_name in product_df.columns:
    if dict(product_df.dtypes)[col_name] == 'string':
        product_df = product_df.withColumn(col_name, regexp_replace(col(col_name), '"', ''))

display(product_df)
print(f"Row count: {product_df.count()}")

product_df.write.mode("overwrite").saveAsTable("fraud_detection.bronze.product_catalog_raw")

In [0]:
print(f"Row count: {product_df.count()}")

product_df.write.mode("overwrite").saveAsTable("fraud_detection.bronze.product_catalog_raw")

display(spark.sql("SELECT COUNT(*) FROM fraud_detection.bronze.product_catalog_raw"))